# Lesson 27 Lab — Production Deployment, Versioning, and Rollback

**Puzzle:** What makes a quantized release safely reversible?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A release unit includes immutable model/tokenizer/recipe/runtime/container identities, metrics, canary policy, observability, and an already verified rollback target.

### Core mechanism

Promotion is a state machine: offline gates -> load/smoke -> shadow -> canary -> broader rollout. Every transition consumes fixed evidence and has an automatic stop/rollback condition.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "27-production-rollout"
device = require_cuda()
torch.manual_seed(2026 + 27)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Slow rollout reduces blast radius but delays benefit; aggressive rollout increases risk. Rollback speed depends on keeping the baseline warm and compatible with current traffic.

### What this code tests

The notebook converts measured CUDA error and timing into a deterministic synthetic release decision and rollback manifest, without claiming live traffic.

**Experiment:** Evaluate a synthetic candidate against frozen gates and emit a release decision plus rollback manifest from measured CUDA output error and timing.

**Declared evidence label:** `capacity-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
w=torch.randn(2048,2048,device=device,dtype=torch.bfloat16); x=torch.randn(16,2048,device=device,dtype=torch.bfloat16); ref=x@w.t(); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); dq=dq.bfloat16(); cand=x@dq.t()
base_t=cuda_benchmark(lambda:x@w.t(),warmup=4,repeats=15); cand_t=cuda_benchmark(lambda:x@dq.t(),warmup=4,repeats=15); err=error_metrics(ref,cand)
gates={"rmse_lte_0_5":err["rmse"]<=0.5,"latency_regression_lte_10pct":cand_t["median_ms"]<=base_t["median_ms"]*1.10}
decision="promote_to_canary" if all(gates.values()) else "rollback"
manifest={"candidate":"reference-int4-v1","baseline":"bf16-v1","decision":decision,"gates":gates,"rollback_target":"bf16-v1"}
result=base_result(27,"capacity-model"); result.update({"baseline_timing":base_t,"candidate_timing":cand_t,"output_error":err,"release_manifest":manifest,
    "conclusion":"Frozen synthetic gates produced a deterministic release or rollback decision; no live service canary was claimed."})


## 3. Inspect the evidence

The manifest is a deployment-control exercise, not evidence that a real service was canaried.

### Acceptance and rollback gate

Version every artifact, define quality/latency/error/capacity thresholds, monitor slices, and test the rollback command before canary traffic.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "baseline_timing": {
    "median_ms": 0.01936,
    "p90_ms": 0.024256,
    "repeats": 15,
    "samples_ms": [
      0.035968,
      0.021824,
      0.020256,
      0.019616,
      0.019168,
      0.019808,
      0.024448,
      0.018944,
      0.01936,
      0.024256,
      0.019168,
      0.018912,
      0.018784,
      0.019072,
      0.018848
    ],
    "warmup": 4
  },
  "candidate_timing": {
    "median_ms": 0.018848,
    "p90_ms": 0.019424,
    "repeats": 15,
    "samples_ms": [
      0.020192,
      0.019072,
      0.018848,
      0.019008,
      0.01904,
      0.018656,
      0.018528,
      0.019424,
      0.019776,
      0.018272,
      0.01856,
      0.018528,
      0.018432,
      0.018944,
      0.018784
    ],
    "warmup": 4
  },
  "conclusion": "Frozen synthetic gates produced a deterministic release or rollback decision; no live service canary was claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce 

## 4. Explain the result

Automate the decision and rollback metadata before exposing traffic; never improvise rollback after a regression.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).